In [ ]:
import json

import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.constants import DATA_PATH, FULL_RUN
from emu_renewal.inputs import get_income_group, get_cont_of_country
from emu_renewal.outputs import get_param_vals_by_analysis, get_ratios_from_disps, get_median_ratios
from emu_renewal.plotting import plot_dispersion_ratio_figure
from emu_renewal.utils import get_analysis_paths, get_country_name

set_matplotlib_formats("svg")

In [ ]:
SEED = 42
ANALYSIS = "oxcgrt_floored"
BASELINE = "no_scaling"
N_EXEMPLARS = 3

all_countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))
analysis_paths = get_analysis_paths(FULL_RUN, all_countries)
avail_countries = [
    iso3 for iso3, paths in analysis_paths.items() if BASELINE in paths and ANALYSIS in paths
]

disp_posts = {}
for iso3 in avail_countries:
    c_paths = {k: analysis_paths[iso3][k] for k in (BASELINE, ANALYSIS)}
    disp_posts[iso3] = get_param_vals_by_analysis("dispersion_proc", c_paths)

np.random.seed(SEED)
ratio_dists = get_ratios_from_disps(disp_posts)
median_ratios = get_median_ratios(ratio_dists, ANALYSIS)

In [ ]:
# One country drawn at random from each tertile of the median ratio
rng = np.random.default_rng(SEED)
ratio_series = pd.Series(median_ratios).sort_values()
tertiles = pd.qcut(ratio_series, N_EXEMPLARS, labels=False)

exemplar_iso3s = []
for tier in range(N_EXEMPLARS):
    pool = ratio_series.index[tertiles == tier].tolist()
    exemplar_iso3s.append(str(rng.choice(pool)))
exemplar_iso3s = list(reversed(exemplar_iso3s))

exemplar_df = pd.DataFrame(
    {
        "iso3": exemplar_iso3s,
        "country": [get_country_name(c) for c in exemplar_iso3s],
        "median_ratio": [median_ratios[c] for c in exemplar_iso3s],
        "tertile": list(reversed(range(1, N_EXEMPLARS + 1))),
    }
)

In [ ]:
fig = plot_dispersion_ratio_figure(
    disp_posts,
    ratio_dists,
    exemplar_iso3s,
    analysis_type=ANALYSIS,
    baseline=BASELINE,
)
display(fig)

In [ ]:
fig.savefig("figure_4.svg", bbox_inches="tight")

World Bank income group of countries with and without a large policy effect. The dispersion ratio is the median of posterior draws of residual process dispersion under no scaling divided by the same quantity under OxCGRT floored scaling. Ratios above 1 indicate that policy scaling reduced unexplained transmission variation. Income groups are the World Bank classification in `data/income/CLASS.xlsx` (2023 GNI per capita), via `get_income_group`. Venezuela is unclassified in that file.

In [ ]:
INCOME_ORDER = [
    "Low income",
    "Lower middle income",
    "Upper middle income",
    "High income",
    "Unclassified",
]
THRESHOLDS = [1.2]

country_ratios = pd.DataFrame(
    {
        "country": [get_country_name(iso3) for iso3 in median_ratios],
        "income": [get_income_group(iso3) for iso3 in median_ratios],
        "median_ratio": pd.Series(median_ratios),
    }
)
country_ratios["continent"] = country_ratios.index.map(get_cont_of_country)
country_ratios["income"] = country_ratios["income"].fillna("Unclassified")
present_incomes = [group for group in INCOME_ORDER if group in set(country_ratios["income"])]
country_ratios["income"] = pd.Categorical(
    country_ratios["income"], categories=present_incomes, ordered=True
)


def _country_list(group: pd.DataFrame) -> str:
    ordered = group.sort_values("median_ratio", ascending=False)
    if ordered.empty:
        return "—"
    return ", ".join(f"{row.country} ({row.median_ratio:.2f})" for row in ordered.itertuples())


def income_count_table() -> pd.DataFrame:
    """Number and share of countries above each dispersion-ratio threshold."""
    rows = []
    grouped = list(country_ratios.groupby("income", observed=True))
    grouped.append(("All", country_ratios))
    for income, group in grouped:
        row = {"income": str(income), "n": len(group)}
        for threshold in THRESHOLDS:
            n_above = int((group["median_ratio"] > threshold).sum())
            row[f"n > {threshold:.1f}"] = n_above
            row[f"share > {threshold:.1f}"] = f"{n_above / len(group):.0%}"
        rows.append(row)
    return pd.DataFrame(rows).set_index("income")


def income_country_table(threshold: float) -> pd.DataFrame:
    """Countries above and at or below one threshold, listed within each income group."""
    rows = []
    for income, group in country_ratios.groupby("income", observed=True):
        above = group.loc[group["median_ratio"] > threshold]
        below = group.loc[group["median_ratio"] <= threshold]
        rows.append(
            {
                "income": str(income),
                "above": f"{len(above)}/{len(group)}",
                "countries above": _country_list(above),
                "countries at or below": _country_list(below),
            }
        )
    return pd.DataFrame(rows).set_index("income")


display(income_count_table())
for threshold in THRESHOLDS:
    print(f"Median dispersion ratio > {threshold:.1f}")
    with pd.option_context("display.max_colwidth", None):
        display(income_country_table(threshold))